# 2.1 Kernels: a few weights that scan a heartbeat

The class problem of this lesson is a heart monitor: from one heartbeat, say whether it
is normal or one of two kinds of abnormal beat, for a patient the model has never seen.
A missed abnormal beat can be a patient who goes home without a diagnosis.

In lesson 1 a frozen encoder did the heavy work: it turned every image into a vector in
which similar images lay close together, and you trained a small head on top. For
heartbeats no such encoder is ready. A beat is 187 numbers, and whatever structure a
model uses, it has to find in those numbers itself.

This notebook gives you the first block for that, the **kernel**. At the end you can:

- say what a kernel does to a heartbeat, compute one of its outputs by hand, and say why
  it needs so few weights;
- tell three ways apart to make an axis shorter (stride, pool, global pool), with the
  shape before and after;
- predict how many steps of the beat one position of a stack of kernels sees, and
  recompute that number yourself;
- read from repeated runs whether a wider view paid on patients the model never saw,
  and say what the experiment can and cannot separate;
- fold a beat into a grid and say which neighbours a kernel then sees.

| section | the question | goal |
|---|---|---|
| the data, the metric | what is a beat, and how do we score a model fairly? | 1.13, 2.7 |
| 2.1.1 | what does a model without kernels reach on unseen patients? | 1.13, 2.9 |
| 2.1.2 | what does a kernel do, and why is it one operation at every rank? | 2.2, 1.6 |
| 2.1.3 | how do you get fewer positions on an axis? | 2.3 |
| 2.1.4 | how far does one position see? | 2.4 |
| 2.1.5 | does a wider view pay on heartbeats? | 2.4, 2.7 |
| 2.1.6 | the same beat as a 2D grid | 2.8 |

The slides of lesson 2 are referred to by number; keep the deck open beside this
notebook.

## The data: heartbeats, by patient

The data is MIT-BIH, a set of 48 half-hour heart recordings (ECGs) made at a hospital in
Boston. Each recording has a number between 100 and 234, the **record**. One record is one
patient, with one exception: records 201 and 202 are the same man.

The recordings are cut into single beats. A heartbeat on an ECG has one tall, sharp spike,
the **R-peak**; the spike and the dips around it, together about 0.1 s, are the **QRS
complex**: the electrical signal that makes the **ventricles** (the two large lower
chambers of the heart that pump the blood out) contract, just before they do. The slower,
lower bump that follows is the **T wave**, the ventricles recovering. Every beat here starts at its R-peak and runs for 1.2 times
the usual time between two beats in that stretch of the recording, so the next R-peak
usually shows just before the zeros; the rest of the 187 values is zeros. The 187 steps of one
cut-out beat are its **window**. At 125 samples a second, 187 steps are 1.5 s.

Three classes:

- **N**: beats conducted normally from above, including bundle branch block beats, which
  can be wide and odd-shaped (validation records 118, 124 and 207 are full of them), so
  wide does not mean V;
- **S**, supraventricular ectopic: *ectopic* means the beat starts somewhere other than
  in the heart's own pacemaker, *supraventricular* means above the ventricles. The beat
  comes too early but often looks much like a normal one;
- **V**, ventricular ectopic: the beat starts in the ventricles themselves, and is wide
  and odd-shaped.

### Three splits, three roles

Lesson 1 had a train split and a test split. From now on there are three, each with one
job. The dataset has a train split and a test split, 22 records each. This notebook splits
train further into train and validation, so there are three splits: train, validation
(`valid` in the code) and test. That makes 44 records: the other 4 of the 48 (102, 104, 107 and 217) are not in the dataset. They are the records of
patients with an artificial pacemaker, which work on this data usually leaves out.

| split | its job | here |
|---|---|---|
| **train** | the weights are fitted on it | 16 records |
| **validation** (`valid`) | you choose between designs on it: which depth, which kernel | 6 other records, held out: 118, 124, 201, 207, 215, 223 |
| **test** | judges your final choice, once, at the end | 22 other records, left alone in this lesson |

Why not choose on test, as lesson 1 did? Every choice you make by looking at a score
fits that score a little. Make twenty choices on test and the test score is no longer a
fair estimate of the next patient. Test has to stay unseen until the choice is made.

The data is split **by record**, so by patient. A beat of a patient the model fitted on is
not a new case: the next beat of the same heart looks almost the same. Split per beat
instead, and every score is higher than the model would reach on the next patient. (The
one exception: record 201 is in validation, 202 of the same man in test.)

The cell below loads train and validation from the course's dataset on the Hugging Face hub
and counts the beats per class. Nothing is trained in it.

In [ ]:
import sys

import heartbeats as hb
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from notebooktester import param
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from vectormesh.components import Conv1D, Conv2D, Pool1D, Pool2D

logger.remove()
logger.add(sys.stderr, level="WARNING")

MAX_ROWS = param(None, test=2000)  # train beats to load; None loads all of them

train, valid = hb.load(max_rows=MAX_ROWS)
print(
    f"train  {len(train):6d} beats  {train.counts()}  from {len(set(train.records))} records"
)
print(f"valid  {len(valid):6d} beats  {valid.counts()}  records {hb.VAL_RECORDS}")
print(f"one beat: {tuple(train.beats[0].shape)}  (S, D): 187 steps, one channel")
print(f"a batch:  (B, 187, 1)")

Read the counts: nine beats in ten are N, and S is rare, a few hundred beats in each
split. Each beat is `(187, 1)`: 187 **steps** along the time axis `S`, and one value per
step, so `D = 1`. A batch stacks beats into `(B, S, D) = (B, 187, 1)`, the shape lesson 1
used for a series (tensor-ladder §2).

The next cell draws 20 beats per class, laid over each other: in the top row all from one
patient, in the bottom row from many patients.

In [ ]:
rng = np.random.default_rng(0)
labels, records = train.labels.numpy(), train.records
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), sharex=True, sharey=True)
t = np.arange(hb.STEPS) / 125  # seconds after the R-peak
for k, name in enumerate(hb.CLASSES):
    of_class = np.flatnonzero(labels == k)
    most = np.bincount(records[of_class]).argmax()  # the patient with most beats of k
    mine = of_class[records[of_class] == most]
    one = rng.choice(mine, min(20, len(mine)), replace=False)
    many = rng.choice(of_class, min(20, len(of_class)), replace=False)
    for row, rows, who in ((0, one, "one patient"), (1, many, None)):
        for r in rows:
            axes[row, k].plot(t, train.beats[r, :, 0], color="C0", alpha=0.25, lw=1)
        who = who or f"{len(set(records[rows]))} patients"
        axes[row, k].set_title(f"{name}: {len(rows)} beats of {who}")
    axes[1, k].set_xlabel("seconds after the R-peak")
for ax in axes[:, 0]:
    ax.set_ylabel("scaled amplitude")
fig.suptitle("Twenty beats per class: one heart (top) against many hearts (bottom)")
plt.tight_layout()
plt.show()

### Look before you design

Every design below is a bet about these pictures (slide 17). Within one heart the beats of
a class lie on top of each other; across hearts they spread. That spread is why the data
is split by patient, and it is what a model has to see through. Before any model, say
which class you would trust a model to find, and from what you see:

- in **N**, a second spike just before the zeros: the next R-peak;
- in **S**, the part before the second spike looks much like an N. Read where the second
  spike falls, in seconds, in the top row of S against the top row of N, and whether it
  falls at one time or at many in the bottom row of S;
- in **V**, the beat itself is wide and has a shape of its own.

Check what you read against this: in the top row the second spike of S sits near 0.38 s, of N near
0.65 s. The next beat came soon after this S beat: most likely a run of S beats, one after
the other, in record 209 (it holds 364 of the 469 S beats of train). Because the window
starts at the R-peak, the interval *before* the beat, which shows that it came too early,
is not in the window; only the interval *after* it is, and in the bottom row of S that
falls at many different times.

Then write down a strategy in one sentence. Is S a shape of its own that a model can
learn, or is it mostly *not N and not V*? If it is the second, what in a design could say
"unlike a normal beat"? Keep the sentence: 2.1.5 shows where S stands, and lesson 3 comes
back to it.

## The metric: recall per class, and its mean

N is about nine in ten beats. A model that always says N is right nine times in ten, an
accuracy of about 90%, and finds no patient at all (slide 22). Accuracy hides the rare
class, so these notebooks read **recall**.

The **recall of a class** is the share of the beats of that class that the model found.
Say the validation set holds 10 S beats and the model calls 3 of them S: the recall of S is
3 / 10 = 0.3. What the model calls the other 7 does not matter for this number, and
neither does how often it says S when the beat was an N (that is precision, slide 22).

The **macro recall** is the mean of the recalls of N, S and V. Every class counts for a
third, however rare, so a missed S weighs as much as a missed N. Always saying N gives a
recall of 1 for N and 0 for S and V: a macro recall of exactly 1/3, whatever the data.

**Before you run the next cell**: 100 beats, 80 N, 10 S and 10 V. A model finds every N,
3 of the S and 9 of the V, and calls the rest N. Write down its accuracy and its macro
recall.

In [ ]:
truth = np.array([0] * 80 + [1] * 10 + [2] * 10)  # 80 N, 10 S, 10 V
model = truth.copy()
model[83:90] = 0  # 7 of the 10 S called N: 3 found
model[99] = 0  # 1 of the 10 V called N: 9 found
for name, guess in (("this model", model), ("always N", np.zeros_like(truth))):
    s = hb.scores(truth, guess)
    recall = {c: round(r, 2) for c, r in s["recall"].items()}
    print(
        f"{name:10s}  accuracy {s['accuracy']:.2f}  recall {recall}  macro recall {s['macro_recall']:.3f}"
    )

The model's accuracy, 0.92, is only twelve points above always N; its macro recall, 0.733,
is forty points above. The macro recall moves with the rare classes, which is what the
heart monitor is for.

### The loss does not see the imbalance

Every model in this notebook trains on plain **cross-entropy** (lesson 1, 03 §3.2), with
every beat weighted the same. For one beat, cross-entropy is `−log p`, where `p` is the
probability the model gave to the right class: 0 when it was sure and right, large when it
gave the right class little. The loss is the mean over the beats. That loss is not the
metric. On train, 91% of the beats are N
and only 1.4% are S, so a model lowers its loss a lot by leaning towards N. The next cell
computes how low: the loss of a model that knows nothing about beats and always gives the
class frequencies as its probabilities, against one that guesses uniformly.

In [ ]:
share = np.bincount(train.labels.numpy(), minlength=3) / len(train)
print(
    "share of train per class",
    dict(zip(hb.CLASSES, share.round(3).tolist(), strict=True)),
)
print(f"loss of guessing 1/3 each       {np.log(3):.3f}")
print(
    f"loss of giving the frequencies  {-(share * np.log(share)).sum():.3f}  <- without looking at the beat"
)

Check the two numbers: guessing 1/3 costs `−log(1/3) = 1.099` on every beat. Giving the
frequencies costs `−log 0.912` on an N beat, `−log 0.014` on an S beat and `−log 0.074` on
a V beat; weighted by how often each occurs, that is `−Σ share × log share`. Without
looking at a single beat, the loss drops from 1.10 to about 0.34. Nothing in
that loss asks for S: getting S wrong costs little, because S is so rare. That is one
reason S will be found so rarely below. A loss that weighs every class by how rare it is
answers this; your study in `exercises/` has it as the axis `class_weight` (0 is the plain
loss of this notebook, 1 weighs every class, not every beat, the same).

## 2.1.1 Baselines, before any kernel

### An encoder, or train the whole model?

In lesson 1 you reached for a pretrained encoder, and it paid: it had been trained on
millions of images or texts, and its vectors already placed similar items close together.
Whether that is the right move depends on three things (goal 2.9):

- **how much data you have**: train holds 33,454 beats, of which only 469 are S. That is
  enough to train a small model of a few thousand weights from scratch, and little for the
  rare class;
- **how close your data is to what the encoder was trained on**: lesson 1's encoders saw
  photos and text. An ECG is one line of 187 numbers; nothing in a photo of a cat teaches
  what an early heartbeat looks like;
- **what a model without an encoder already reaches**: the baselines below. If a lookup of
  the nearest raw beat already scores well, the raw numbers carry structure a small model
  can use.

You would reach for an encoder when your data is close to its pretraining data (photos
for an image encoder, sentences for a text encoder), or when you have few labelled
examples and an encoder trained on a lot of similar data exists, for ECGs one pretrained on
many hours of heart recordings. Here the second half holds for S, with few labels, but no
ECG encoder is at hand in this course, and lesson 1's encoders are far from ECGs; so this
lesson trains the whole model.

### Four models without a kernel

Lesson 1's order (goal 1.13): a floor that trains nothing, then the simplest trained
model, then one bet at a time. Four models without a kernel:

- **always N**: macro recall 1/3, as computed above;
- **1-nearest-neighbour on the raw beat**: lesson 1 compared an encoder's vectors by cosine
  similarity and gave each test item the label of the closest training item. Here there is
  no encoder, so the raw beat of 187 steps is its own vector (goal 2.9): is a beat of an
  unseen patient closest to a beat of the same class? It trains nothing;
- **a linear layer on the 187 steps**: lesson 1's simplest head, `(B, 187) -> (B, 3)`;
- **one hidden layer of 256 with a GELU**: lesson 1's head with a hidden layer, on the
  raw steps.

**The floor** every design in this lesson must clear is the best of these. **Before you
run the next cell**, write down what you expect 1-nearest-neighbour to reach: above or
below 1/3, and which class you expect it to find best.

In [ ]:
always_n = hb.scores(valid.labels.numpy(), np.zeros(len(valid), dtype=int))
print(f"{'always N':30s} macro recall {always_n['macro_recall']:.3f}")

knn = KNeighborsClassifier(n_neighbors=1, metric="cosine").fit(
    train.beats[:, :, 0], train.labels
)
nearest = hb.scores(valid.labels.numpy(), knn.predict(valid.beats[:, :, 0]))
recall = {c: round(r, 2) for c, r in nearest["recall"].items()}
print(
    f"{'1-nearest-neighbour, cosine':30s} macro recall {nearest['macro_recall']:.3f}  recall {recall}"
)

### The two heads, and how their results are read

`Flat` is lesson 1's head on the raw steps. It drops the channel axis, `(B, 187, 1) ->
(B, 187)`, so each beat is one vector of 187 numbers, and maps it to three **logits**, one
score per class. The drawing below has the shape on every arrow, as lesson 1 drew its
heads (04 §4.1); `diagrams.py` is lesson 1's drawing code.

Both heads train for `EPOCHS` epochs with Adam at a learning rate of 0.001, as does every
trained model in this notebook. Each is trained `REPEATS` times, from
other random weights each time, because one run alone does not say how much of its score
is luck (lesson 1, 04 §4.0). `hb.train` scores the model on validation after every epoch;
the **score of a run** is the mean validation macro recall of its last 3 epochs, because
the score of one epoch alone swings by a few hundredths from one epoch to the next. The
**recall per class** that the cells print is that of the last epoch, averaged over the
runs.

`hb.line` prints one result in the unit lesson 1's `compare` used (05), for example

```
a design                             macro recall 0.500 ± 0.010     5.0s   -2.6 se
```

(made-up numbers):

- `0.500 ± 0.010` is the mean score of the runs and their spread (the std);
- `5.0s` is the training time of one run;
- `-2.6 se` is the difference with the result it is compared to, in **standard errors**
  of that difference: `(mean_a − mean_b) / sqrt(std_a² / n_a + std_b² / n_b)`. A
  difference within about 2 se is no result: other random weights could flip it.

With 4 runs per arm the 2 se line is a rule of thumb: a difference near 2 to 3 se is
marginal, one well beyond it is clear.

The standard error answers one question: would other random weights flip the result *on
these 6 validation patients*? It says nothing about how the result carries over to other
patients; six hearts are a small sample of hearts.

1-nearest-neighbour trains nothing, so it has no spread: its std is 0, and against it the
standard error comes from the trained design's own spread alone.

**Why 20 epochs.** A comparison is only fair when every arm has finished learning (lesson 1,
05 E3). The curves under the results show whether they did: `hb.curves` draws the mean
over the runs as a line, and shades the range of the runs, from the lowest to the highest,
around it. **Before you run the next
cell**, write down which of the two heads you expect to end above 1-nearest-neighbour.

In [ ]:
sys.path.insert(0, "../1_representation_and_head")  # lesson 1's block diagrams
import diagrams
from diagrams import Block, draw_architecture

# the kinds of lesson 1's drawings, named for the blocks of this lesson
diagrams.LEGEND.update(
    data="the beat",
    proj="reshape (no weights)",
    noise="pool (no weights)",
    hidden="kernel or hidden layer, then GELU",
    layer="output Linear",
)


def draw(rows: dict[str, list[Block]]) -> None:
    # one row per model, the shape on every arrow, all rows at one scale
    diagrams.draw_architectures(rows)
    plt.show()


EPOCHS = param(20, test=1)
REPEATS = param(4, test=1)


class Flat(nn.Module):
    # lesson 1's head on the raw steps: (B, 187, 1) -> (B, 187) -> (B, 3)

    def __init__(self, hidden: int = 0):
        super().__init__()
        layers = (
            [nn.Linear(hb.STEPS, hidden), nn.GELU(), nn.Linear(hidden, len(hb.CLASSES))]
            if hidden
            else [nn.Linear(hb.STEPS, len(hb.CLASSES))]
        )
        self.head = nn.Sequential(*layers)

    def forward(self, x):
        return self.head(x[..., 0])  # x[..., 0] drops the channel axis


BEAT = Block("beat", "(B, 187, 1)", "data")
DROP = Block("x[..., 0]", "(B, 187)", "proj")
draw(
    {
        f"linear layer: {hb.n_params(Flat()):,} weights": [
            BEAT,
            DROP,
            Block("Linear\n187 → 3", "(B, 3)"),
        ],
        f"one hidden layer: {hb.n_params(Flat(256)):,} weights": [
            BEAT,
            DROP,
            Block("Linear 187 → 256\n+ GELU", "(B, 256)", "hidden"),
            Block("Linear\n256 → 3", "(B, 3)"),
        ],
    }
)

# 1-nearest-neighbour as a result of one run without spread, so hb.line can compare to it
floor = hb.Spread([hb.Run(nearest, 0.0, [], [], [nearest["macro_recall"]])])

linear = hb.spread(lambda: Flat(), train, valid, EPOCHS, REPEATS)
hidden = hb.spread(lambda: Flat(hidden=256), train, valid, EPOCHS, REPEATS)
print(f"{'1-nearest-neighbour (the floor)':36s} macro recall {floor.mean:.3f}")
print(hb.line("linear layer", linear, versus=floor))
print(hb.line("one hidden layer of 256", hidden, versus=floor))
for name, result in (("linear layer", linear), ("one hidden layer", hidden)):
    print(
        f"  recall per class, {name:17s}",
        {c: round(r, 2) for c, r in result.recall().items()},
    )

In [ ]:
hb.curves(
    {"linear layer": linear, "one hidden layer of 256": hidden},
    "Lesson 1's heads on the raw steps; dashed: 1-nearest-neighbour",
    floor=floor.mean,
)

_Reading_ (numbers from the run of this notebook; yours differ in the last digit).

**The linear layer says N.** Its macro recall stays near 1/3 in every epoch of the right
panel and ends at 0.344, and its recall per class is 0.99 for N, 0 for S and 0.05 for V: it
calls nearly every beat N. Its train loss does fall, from about 0.4 to about 0.17, below
the 0.34 that the class frequencies alone give, and it is still sinking slowly at epoch 20.
It learned something about the beats, but not enough to move S or V over the line. Two
things meet here. One linear layer gives each class a weighted sum of the 187 raw
amplitudes, one fixed weight per step. Every beat starts at its R-peak, so what moves
between hearts is the rest: the T wave (the bump after the spike) and the next R-peak
land at other steps, and the
amplitudes differ. Such a shift moves that sum as much as a real difference in shape
does. And the plain loss, with nine beats in ten N, rewards leaning
on N.

**The hidden layer** ends at 0.489: 93% of N, 52% of V, and no S. Its train loss falls
fast and flattens; its validation recall stops rising after about seven epochs and moves
within a band of about ±0.02 after that, so more epochs would not change the comparison.

The linear layer's −144.9 se is huge because neither arm has much spread: the linear runs
agree to 0.003 and 1-nearest-neighbour has none, so the standard error is tiny. It says
the gap is not luck, not that it is large; the gap itself is 0.18.

**The floor is 1-nearest-neighbour.** It trains nothing and still ends above the hidden
layer, by 7.0 standard errors. That is what the numbers show at this training length and
with this loss; it does not say that a trained head cannot do better, only that this one
did not. Every design below is read against 1-nearest-neighbour, and it has to clear it
to count as having learned more than a lookup of the closest beat.

For the encoder question this is the third number: without any encoder, the nearest raw
beat already reaches 0.526, well above 1/3. The raw beats carry structure; the rest of
this notebook asks how much of it a model trained from scratch can use.

## 2.1.2 What a kernel does

A **kernel** is a handful of weights that slides along an ordered axis. At every position
it multiplies the values under it with its weights and sums them; that sum is one value of
the output. The **same** weights are used at every position (slide 5). Sliding a kernel
along an axis is called a **convolution**; lesson 1 ran one (02 §2.2).

Try it by hand first. The kernel `[-1, 0, 1]` on the five numbers `[1, 2, 4, 7, 11]`: at
the first position it reads `1, 2, 4` and gives `−1·1 + 0·2 + 1·4 = 3`. **Write down the
other outputs before you run the next cell.** How many are there?

Then the edges. Without help, the kernel of 3 only fits on 3 of the 5 positions, so the
output is shorter than the input (lesson 1 saw the same: "no padding costs two
positions"). **Padding** adds zeros at both ends, here one on each side, `[0, 1, 2, 4, 7,
11, 0]`, so the kernel fits at every position and the output keeps the input's length. A
kernel of size `k` keeps the length with `k // 2` zeros on each side. Write down the
padded output too.

In [ ]:
numbers = torch.tensor([1.0, 2.0, 4.0, 7.0, 11.0]).reshape(
    1, 5, 1
)  # (B, S, D) = (1, 5, 1)
for padding in (0, 1):
    edge = Conv1D(in_size=1, out_size=1, kernel_size=3, padding=padding)
    with torch.no_grad():
        edge.conv.weight[:] = torch.tensor([-1.0, 0.0, 1.0]).reshape(1, 1, 3)
        edge.conv.bias.zero_()
    out = edge(numbers)
    print(
        f"padding {padding}: in {tuple(numbers.shape)} -> out {tuple(out.shape)}   {out[0, :, 0].tolist()}"
    )

Without padding: `[3, 5, 7]`, three outputs, each the difference of the two neighbours:
how fast the numbers climb. With padding the length stays 5, and the two edge values, `2`
and `−7`, are what the kernel makes of the zeros: an edge that is not in the data.

### `Conv1D` keeps the channels last

Lesson 1 ran PyTorch's `nn.Conv1d`, which wants the channels second, `(B, C, L)`, and
permuted a series into that shape first (tensor-ladder §2). Every sequence in this course
is stored `(B, S, D)`, with the channels last. vectormesh's `Conv1D` takes `(B, S, D)` as
it is, swaps the axes inside, and gives `(B, S', D')` back: `S'` is the new length, `D'` the
number of kernels. Each kernel gives one output line, one **channel**, so `out_size=16`
means 16 kernels and 16 channels. `Conv2D` keeps images as `(B, C, H, W)`, channels
second, as images are stored in this course.

Below, kernels set by hand on one V beat: two that smooth, and two that answer where the
beat changes. The last two share one `Conv1D` with `out_size=2`, so they come out as two
channels of one output.

In [ ]:
def by_hand(*weights: list[float]) -> Conv1D:
    # one kernel per row of `weights`, set by hand, no bias: (B, S, 1) -> (B, S, rows)
    size = len(weights[0])
    conv = Conv1D(in_size=1, out_size=len(weights), kernel_size=size, padding=size // 2)
    with torch.no_grad():
        conv.conv.weight[:] = torch.tensor(weights).unsqueeze(1)
        conv.conv.bias.zero_()
    return conv


beat = train.beats[np.flatnonzero(train.labels.numpy() == 2)[0]]  # a V beat, (187, 1)
x = beat.unsqueeze(0)  # (1, 187, 1)
mean3 = by_hand([1 / 3] * 3)(x)
mean15 = by_hand([1 / 15] * 15)(x)
change = by_hand([-1.0, 0.0, 1.0], [1.0, -2.0, 1.0])(x)
print(f"two kernels in one Conv1D: in {tuple(x.shape)} -> out {tuple(change.shape)}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t, beat[:, 0], color="0.6", lw=3, label="original")
axes[0].plot(t, mean3[0, :, 0].detach(), label="mean of 3")
axes[0].plot(t, mean15[0, :, 0].detach(), label="mean of 15")
axes[0].set_title("Smoothing: the mean over 3 and over 15 neighbours")
axes[0].set_ylabel("scaled amplitude")
axes[1].axhline(0, color="0.85", lw=1)
axes[1].plot(t, change[0, :, 0].detach(), label="edge [-1, 0, 1]")
axes[1].plot(t, change[0, :, 1].detach(), label="curvature [1, -2, 1]")
axes[1].set_title("Change: where the beat rises or falls, and where it bends")
axes[1].set_xlabel("seconds after the R-peak")
axes[1].set_ylabel("kernel output")
for ax in axes:
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

The mean smooths. Over 3 neighbours it barely changes the beat. Over 15 neighbours, 0.12 s
and about the length of a QRS complex, both peaks flatten: the R-peak at the very start
drops from 0.75 to about 0.15 (there, half of the kernel's 15 steps are padding zeros), and the tall
wave near 0.5 s from above 0.8 to below 0.6, a hump.

The edge kernel `[-1, 0, 1]` is positive where the beat climbs (towards the wave near
0.5 s) and negative where it drops (right after the R-peak, and after the wave). Its value
at 0 s is the padding's: it compares the first step with a zero, as `2` did in the hand
example. Its mirror `[1, 0, -1]` draws the same line upside down, so a layer that learns
either one has learned the same thing. The curvature kernel `[1, -2, 1]` is the edge of the
edge: it answers where the slope itself changes. By far its largest value, about −0.8,
is at 0 s, where the padding zero meets the R-peak. Smaller spikes, below 0.25 either way,
mark the bottom of the dip after the R-peak (about 0.03 s), the fall after the tall wave
(about 0.52 s) and the drop to zeros (about 0.63 s); between them it flickers near zero. A trained layer learns its own weights, but every one of them does this:
it looks at a few neighbours, with the same weights everywhere.

### Why so few weights

A dense layer from 187 steps to 187 steps has a weight for every pair of positions. A
kernel of 3 has 3, reused at every position, and it only looks at neighbours (slide 3).

When the input has several channels, one kernel reads all of them. A kernel of size `k`
on `D` input channels is a `D × k` block of weights, one row of `k` per channel: at each
position it multiplies the `k` steps of all `D` channels with their weights and sums
everything, over steps and channels, into one number, plus one bias. A small case: 3 steps
of 2 channels, channel 0 is `[1, 2, 3]` and channel 1 is `[10, 20, 30]`; the kernel has the
row `[1, 0, −1]` for channel 0 and `[0, 1, 0]` for channel 1. Its one output is
`(1·1 + 0·2 − 1·3) + (0·10 + 1·20 + 0·30) = −2 + 20 = 18`. The cell checks it.

So a kernel on `D` channels has `D × k` weights and one bias, and a layer of `D'` kernels
has `D' × (D × k + 1)`. **Before you run the next cell**, compute that number for 32 kernels
of 5 on 32 channels, a layer that 2.1.5 uses.

In [ ]:
two = torch.tensor([[[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]]])  # (B, S, D) = (1, 3, 2)
one = Conv1D(in_size=2, out_size=1, kernel_size=3)
with torch.no_grad():
    one.conv.weight[:] = torch.tensor(
        [[[1.0, 0.0, -1.0], [0.0, 1.0, 0.0]]]
    )  # (kernels, D, k)
    one.conv.bias.zero_()
print(f"one kernel on 2 channels: in {tuple(two.shape)} -> {one(two).item():.0f}")

dense = nn.Linear(hb.STEPS, hb.STEPS)
kernel = Conv1D(1, 1, kernel_size=3, padding=1)
wide = Conv1D(32, 32, kernel_size=5, padding=2)
print(
    f"dense 187 -> 187:            {hb.n_params(dense):6d} weights = 187 x 187 and 187 biases"
)
print(
    f"one kernel of 3:             {hb.n_params(kernel):6d} weights (3 and a bias), same output shape"
)
print(
    f"32 kernels of 5 on 32 chans: {hb.n_params(wide):6d} weights = 32 x (32 x 5 + 1)"
)
print(
    f"  shape {tuple(wide.conv.weight.shape)}: (kernels, input channels, kernel size)"
)

### The same operation at another rank

On an image `(B, C, H, W)` nothing new happens: the kernel is 3 by 3 and slides along two
axes instead of one (slide 6). The cell below puts two hand-set kernels on a 9-by-9 cross.
The first has the rows `[-1, 0, 1]`, three times: in every row it subtracts the pixel on
the left from the pixel on the right. The second is the same kernel turned a quarter,
which subtracts the pixel above from the pixel below.

In [ ]:
image = torch.zeros(1, 1, 9, 9)  # (B, C, H, W)
image[0, 0, 4, 1:8] = 1.0  # the horizontal bar of a cross
image[0, 0, 1:8, 4] = 1.0  # the vertical bar
conv2 = Conv2D(in_size=1, out_size=2, kernel_size=3, padding=1)
with torch.no_grad():
    edge = torch.tensor([[-1.0, 0.0, 1.0]] * 3)
    conv2.conv.weight[:] = torch.stack([edge, edge.T]).unsqueeze(1)
    conv2.conv.bias.zero_()
maps = conv2(image)
print(
    f"in {tuple(image.shape)} -> out {tuple(maps.shape)}: 9 weights per kernel, two kernels"
)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, img, title in zip(
    axes,
    [image[0, 0], maps[0, 0], maps[0, 1]],
    ["input: a cross", "rows [-1, 0, 1]: vertical edges", "turned: horizontal edges"],
    strict=True,
):
    shown = ax.imshow(img.detach(), cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.colorbar(shown, ax=axes, shrink=0.8, label="red positive, blue negative, white 0")
plt.show()

Red is positive, blue negative, white zero. The first kernel compares each pixel's right
neighbours with its left ones. Along the vertical bar the picture changes from left to
right, from 0 to 1 and back to 0, so the output is red just left of the bar (the bar lies
to its right) and blue just right of it. Along the horizontal bar nothing changes from
left to right except at its two ends, so the output there is almost white. Rows of
`[-1, 0, 1]` find **vertical** edges; turned a quarter, the kernel finds horizontal ones.
It is the edge kernel of the beat, with one more axis to slide along.

### Two kernels without an activation are one kernel

Lesson 1 (02 §2.1, goal 1.6) showed that a stack of linear layers without a nonlinearity is
one linear layer. A kernel is linear too, so the same holds. Take a kernel `[1, 2]` and on
its output a kernel `[3, 1]`. One output of the second reads two outputs of the first:

`3 · (1·x₀ + 2·x₁) + 1 · (1·x₁ + 2·x₂) = 3·x₀ + 7·x₁ + 2·x₂`

That is one kernel of 3, `[3, 7, 2]`: every weight is a sum of products of a weight of the
first kernel and one of the second. `np.convolve` computes exactly those sums. The cell
checks the small example, then a kernel of 3 on a kernel of 5 with random weights against
the one kernel of 7 built from them.

In [ ]:
print("[1, 2] then [3, 1] is one kernel", np.convolve([1, 2], [3, 1]))

k3, k5, k7 = Conv1D(1, 1, 3), Conv1D(1, 1, 5), Conv1D(1, 1, 7)  # random weights
with torch.no_grad():
    for k in (k3, k5, k7):
        k.conv.bias.zero_()
    w7 = np.convolve(k5.conv.weight[0, 0].numpy(), k3.conv.weight[0, 0].numpy())
    k7.conv.weight[0, 0] = torch.tensor(w7.copy())
x = train.beats[:4]
stacked, single = k3(k5(x)), k7(x)
print(
    f"kernel 5 then 3: {tuple(stacked.shape)};  one kernel of 7: {tuple(single.shape)}"
)
print(
    f"largest difference between the two outputs: {(stacked - single).abs().max():.1e}"
)

The two outputs agree up to rounding (the largest difference is below 1e-7). Without padding each kernel
costs `k − 1` positions: 187 − 4 − 2 = 181, and the one kernel of 7 costs the same 6.
Depth only buys something with an activation between the kernels, so every stack below
puts a GELU after each kernel.

## 2.1.3 Fewer positions on an axis

Downsampling is nothing more than fewer positions on an axis (slide 7). Three ways:

- a kernel with **stride 2** jumps two positions each time; its weights decide how two
  positions become one: learned;
- a **pool** of 2 keeps the larger (max pool) or the mean of each pair: a fixed rule, no
  weights;
- a **global pool** takes the whole axis into one value per channel: an aggregation, and
  the axis is gone. Lesson 1's mean over chunks was one.

**Before you run the next cell**, write down the shape after each of the three, starting
from 8 beats after a kernel of 5 with 16 channels, `(8, 187, 16)`.

In [ ]:
x = train.beats[:8]  # (B, S, D) = (8, 187, 1)
x16 = Conv1D(1, 16, 5, padding=2)(x)
print(f"input                       {tuple(x.shape)}")
print(f"kernel of 5, 16 channels    {tuple(x16.shape)}")
print(
    f"kernel of 5 with stride 2   {tuple(Conv1D(16, 16, 5, stride=2, padding=2)(x16).shape)}"
)
print(
    f"max pool of 2               {tuple(Pool1D(2)(x16).shape)}   <- the odd last step is dropped"
)
print(
    f"global mean pool            {tuple(x16.mean(dim=1).shape)}   <- the S axis is gone"
)

## 2.1.4 How far one position sees

The **receptive field** of an output position is the number of input steps it depends on
(slide 8). A kernel of 5 sees 5. A second kernel of 5 on top sees 5 positions of the first,
each of which saw 5, overlapping: 9 in all. A pool of 2 makes every position above it
stand for two positions below, so every later kernel takes steps twice as large on the
input.

The rule, which you can apply to any stack from the bottom up. Keep two numbers: the
**field**, which starts at 1, and the **jump**, the number of input steps between two
neighbouring positions of the current layer, which starts at 1.

- a kernel of size `k` adds `(k − 1) × jump` to the field;
- a pool of 2 adds `(2 − 1) × jump` to the field, and then doubles the jump.

Every block of the stacks below is a kernel of 5, a GELU, and (with pools) a max pool of 2.
A GELU changes no field. Worked out:

| depth | step | field | jump |
|---|---|---|---|
| | start | 1 | 1 |
| 1 | kernel of 5: + 4 × 1 | 5 | 1 |
| 1 | pool of 2: + 1 × 1, jump doubles | **6** | 2 |
| 2 | kernel of 5: + 4 × 2 | 14 | 2 |
| 2 | pool of 2: + 1 × 2, jump doubles | **16** | 4 |

Without pools the jump stays 1, so every kernel of 5 adds 4: `1 + 4 × depth`.

**Before you run the next cell**: compute the field at depth 3 with pools. Then: a beat is
187 steps. How many blocks do you need before one position sees the whole beat, without
pools, and with them?

`hb.stack(depth, pool=...)` builds the blocks as a list of layers, and
`hb.receptive_field` applies the rule above to it. The cell prints the layers of
`hb.stack(2)` first, so you can follow the rule along them.

In [ ]:
for layer in hb.stack(2):
    print("  ", layer)
print()
print(f"{'depth':>5} {'field, no pools':>16} {'field, pool of 2':>17}")
for depth in range(1, 7):
    print(
        f"{depth:5d} {hb.receptive_field(hb.stack(depth, pool=False)):16d} {hb.receptive_field(hb.stack(depth)):17d}"
    )

Without pools, 47 kernels of 5 are needed to see 187 steps; with pools, 5 blocks see 156
and 6 see more than the beat. The pools let the field grow by doubling, not by adding.

The figure below puts the table on the V beat of 2.1.2. Take one output position of the
stack with pools, the one whose field is centred near 0.4 s, and shade the input steps
that can move it at all. `hb.reach` finds them by asking, for every input step, whether a
small change of that step changes the output; outside the field nothing does, whatever the
weights.

In [ ]:
CENTRE = 50  # input step 50, 0.4 s after the R-peak
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(t, beat[:, 0], color="0.3", lw=1.5, label="the V beat")
for colour, depth in zip(("C2", "C1", "C0"), (5, 3, 1), strict=True):  # widest first
    features = nn.Sequential(*hb.stack(depth))
    field = hb.receptive_field(list(features))
    seen = t[hb.reach(features, CENTRE) > 0]
    label = f"depth {depth}: field {field} steps, {field / 125:.2f} s"
    ax.axvspan(
        seen.min(), seen.max(), color=colour, alpha=0.25, ec=colour, lw=2, label=label
    )
ax.set_title("What one output position, centred near 0.4 s, sees of the beat")
ax.set_xlabel("seconds after the R-peak")
ax.set_ylabel("scaled amplitude")
ax.legend(loc="upper right")
plt.show()

Depth 1 sees 6 steps, 0.05 s: a sliver of the dip between the two waves of this V beat.
Depth 3 sees 36 steps, 0.29 s: from the top of the first wave to past the second. Depth 5
has a field of 156 steps, 1.25 s, but its field runs off the start of the beat, so the
shaded part, from 0 to 1.0 s, is shorter than the field; it holds the R-peak, both waves and
the zeros after the beat. Whether features formed over wider fields help to tell N, S
and V apart is a question for training, the next section.

## 2.1.5 Does a wider field per feature pay?

### What a kernel bets on

Every block assumes something about the data; that assumption is its **inductive bias**,
a bet you make when you choose the block (slide 18, goal 2.7). A kernel makes two bets:

- **locality**: steps next to each other belong together, so a kernel looks only at a few
  neighbours;
- **weight sharing**: a pattern means the same wherever it occurs, so the same weights
  are used at every position.

Both pay when the shape of a beat is built from small local pieces, an edge here, a bend
there, that can sit anywhere. The stacks below end in a **flatten**: `(B, S', D') ->
(B, S' × D')`, every position's channels laid end to end in one vector, and a linear layer
from that vector to three logits. That head gives every position its own weights again:
it knows *where* a pattern was found. The kernels share; the head does not.

### What the receptive field means here

The head reads every position of the last layer, so in every model below the output
depends on all 187 steps: even at depth 1 the model as a whole sees the whole beat. The
receptive field is the field of **one feature**: the stretch of the beat that a kernel and
its GELU combine, nonlinearly, into one number, before the head adds all those numbers up
in one linear step. Depth 1 builds its features from fields of 6 steps, depth 3 from
fields of 36. So the question of this section is: do features formed over wider fields
pay, when a linear readout of the whole beat follows?

### The design

`hb.Kernels(depth)` is the stack of 2.1.4 with that head:
`(B, 187, 1) -> (B, S', 32) -> (B, S' × 32) -> (B, 3)`, 32 kernels per layer. It lives in
`heartbeats.py` because notebook 2.2 imports the same class as its result to beat. The
drawing shows it at depth 1 and 3, and a **control** that the next paragraph explains.

In [ ]:
def kernel_blocks(model: nn.Module) -> list[Block]:
    # the blocks of a hb.Kernels model, with the shape its own layers give
    x, blocks = torch.zeros(1, hb.STEPS, 1), [BEAT]
    with torch.no_grad():
        for layer in model.features:
            x = layer(x)
            shape = f"(B, {x.shape[1]}, {x.shape[2]})"
            if isinstance(layer, Conv1D):
                blocks.append(
                    Block(
                        f"kernel {layer.kernel_size}\n{layer.in_size} → {layer.out_size}",
                        shape,
                        "hidden",
                    )
                )
            elif isinstance(layer, nn.GELU):
                blocks[-1] = Block(blocks[-1].label + ", GELU", shape, "hidden")
            elif isinstance(layer, Pool1D):
                blocks.append(Block(f"max\npool {layer.kernel_size}", shape, "noise"))
    flat = x.flatten(1).shape[1]
    return blocks + [
        Block("flatten", f"(B, {flat})", "proj"),
        Block(f"Linear\n{flat} → 3", "(B, 3)"),
    ]


ARMS = {
    "depth 1": lambda: hb.Kernels(1),
    "control: depth 1, kernel 35": lambda: hb.Kernels(1, kernel_size=35),
    "depth 3": lambda: hb.Kernels(3),
    "depth 5": lambda: hb.Kernels(5),
}
draw(
    {
        name: kernel_blocks(ARMS[name]())
        for name in ("depth 1", "control: depth 1, kernel 35", "depth 3")
    }
)

print(f"{'':28s} {'field':>5} {'kernels':>8} {'head':>6} {'total':>7}  GELUs")
for name, make in ARMS.items():
    model = make()
    field = hb.receptive_field(list(model.features))
    gelus = sum(isinstance(m, nn.GELU) for m in model.features)
    print(
        f"{name:28s} {field:5d} {hb.n_params(model.features):8d} {hb.n_params(model.head):6d}"
        f" {hb.n_params(model):7d}  {gelus}"
    )

Read the arrows of depth 1: the kernel keeps 187 steps (padding), the pool halves them to
93, and the flatten lays 93 positions of 32 channels end to end, 2,976 numbers, into the
head. At depth 3 the beat is 23 positions by the end, so the head is much smaller, while
the kernels have more weights: the second and third layer read 32 channels each, `32 ×
(32 × 5 + 1)` weights per layer.

So depth changes more than the field. From depth 1 to depth 3 the field grows from 6 to
36, and also: the kernels' weights grow, the head's shrink, the number of positions that
reach the head drops from 93 to 23, and the number of nonlinearities (GELUs) goes from 1
to 3. A difference between depth 1 and depth 3 could come from any of these.

**The control** separates the field from the depth. One kernel of 35 and a pool give a
field of `1 + 34 + 1 = 36`, the field of depth 3 (the table prints it), with one layer, one
GELU, 93 positions into the head, and the head of depth 1. It is not a perfect control
either: against depth 1 its kernels have 1,152 weights instead of 192. Read the three
against each other:

- control ≈ depth 3, both above depth 1: one wide kernel gets what depth gets;
- depth 3 above the control: depth (more layers, more nonlinearities) adds something one
  wide kernel does not;
- control above depth 3: one wide kernel pays, and the extra layers may cost something.
  Two candidates: the pools throw positions away (depth 3 sends 23 positions into the
  head, the control 93), and a deeper stack can be harder to train in the same number of
  epochs;
- control ≈ depth 1: one wide kernel does not pay.

Against depth 1 the control changes the field and the kernel weights together, so a gain
of the control says that a wider kernel paid; it cannot say whether the field or the
extra kernel weights did it.

**Before you run the next cell**, write down what you expect: does the macro recall rise
from depth 1 to depth 3, and on to depth 5? Where will the control land? And where do you
expect the recall of S?

In [ ]:
kernels = {
    name: hb.spread(make, train, valid, EPOCHS, REPEATS) for name, make in ARMS.items()
}
for name, result in kernels.items():
    print(hb.line(name, result, versus=floor))
print()
CONTROL = "control: depth 1, kernel 35"
for a, b in (
    ("depth 3", "depth 1"),
    (CONTROL, "depth 1"),
    ("depth 3", CONTROL),
    ("depth 5", "depth 3"),
    ("depth 5", CONTROL),
):
    print(
        f"{a:28s} against {b:28s} {hb.standard_errors(kernels[a], kernels[b]):+5.1f} se"
    )
print()
for name, result in kernels.items():
    print(
        f"recall per class, {name:28s}",
        {c: round(r, 2) for c, r in result.recall().items()},
    )
print(
    f"recall per class, {'1-nearest-neighbour':28s}",
    {c: round(r, 2) for c, r in nearest["recall"].items()},
)
print()
s_records, s_counts = np.unique(
    train.records[train.labels.numpy() == 1], return_counts=True
)
print(
    "S beats of train per record:",
    dict(zip(s_records.tolist(), s_counts.tolist(), strict=True)),
)

In [ ]:
hb.curves(
    kernels,
    "Kernel stacks of depth 1, 3, 5 and the control; dashed: 1-nearest-neighbour",
    floor=floor.mean,
)

_Reading_ (numbers from the run of this notebook; yours differ in the last digit).

**Did every arm finish?** The train losses flatten from about epoch 10 on and keep sinking
slowly. The validation recall of depth 3 and depth 5 is flat from about epoch 6, the
control's from about epoch 15; depth 1 rises slowest and still drifts up by a hundredth or
two after epoch 15. Twenty epochs are enough to compare the levels; for depth 1 the number
may be a little low.

**A wider kernel.** Depth 1 (field 6) ends at 0.486, below 1-nearest-neighbour by 12.9 se.
Depth 3 (field 36) ends at 0.538, 8.2 se above depth 1, and 2.1 se above the floor:
marginal, so whether depth 3 beats 1-nearest-neighbour is not settled. The control, one
kernel of 35 at depth 1, ends at 0.559: 16.8 se above depth 1. A wider kernel paid, and it
needed no depth to do so. What this run cannot say is why: against depth 1 the control has
both a wider field (36 against 6) and six times the kernel weights (1,152 against 192), and
the two are not separated here.

**Depth 3 against the control.** The control is 3.3 se above depth 3, the third case
above. But depth 5, with more layers still and only 5 positions into the head, is 3.9 se
above depth 3 and level with the control (1.1 se, no result). That weakens both candidates:
fewer positions into the head and a harder training for a deeper stack both apply more to
depth 5 than to depth 3, and depth 5 did not trail. So the run does not show that depth
itself costs; depth 3 trailing the control is a small result with 4 runs per arm, without
an explanation this output supports.

**Depth 5** (field 156, most but not all of the 187 steps) ends at 0.564, 9.8 se above the
floor: as good as one kernel of 35, not clearly better.

**S.** Read the recall per class: every kernel arm finds 98–99% of N and 48–69% of V,
and at most 3% of S. 1-nearest-neighbour finds 2% of S as well. The wider kernels lifted
V, not S. Three explanations fit this output, and it does not choose between them:

- **rarity under a plain loss**: S is 1.4% of train, and the plain cross-entropy barely
  charges a model for missing it (the metric section). But 1-nearest-neighbour has no loss
  at all and finds no more S, so rarity under the loss cannot be the whole story;
- **what makes S odd lies mostly outside the window**: an S beat comes too early, relative
  to the beat before it. The window starts at the R-peak, so that interval is not in it;
  only the interval after the beat is (where the second spike falls), and that varies from
  heart to heart. A kernel, however wide its field, cannot see the prematurity itself;
- **a shift between patients**: the last line counts the S beats of train per record.
  364 of the 469 come from record 209 and 78 from record 220: S in train is mostly two
  hearts. The S beats of the six validation patients may look unlike those two, and the
  interval after the beat, which differs from heart to heart, is one way they can differ.

A study can test the first: weigh the loss by class (`class_weight`) and see whether S
rises. In your own study for lesson 2 (`exercises/study.py`, walked through in
`docs/your-study.md`), `kernel_size` is an axis, and so is `parallel`: a second, wider kernel
beside the first, their channels concatenated (slide 8). The control above is a reason to
try a wide kernel there.

## 2.1.6 The same beat as a 2D grid

187 is 11 times 17. Fold the beat row by row into a grid `(B, 1, 11, 17)`: steps 0 to 16
form the first row, 17 to 33 the second, and so on. A 3-by-3 kernel on the grid sees,
besides the steps just before and after, the steps one row above and below: 17 steps
earlier and later (slide 19; slide 4 shows this beat). The data does not change; the
shape does, and with it which steps count as neighbours (goal 2.8).

Two things change at once:

- **new neighbours**: a step and the steps 16, 17 and 18 away now sit under one 3-by-3 kernel. That
  is a bet that what happens 17 steps (0.14 s) later belongs with what happens now;
- **the row wrap**: step 16 ends one row and step 17 starts the next. On the series they
  are neighbours; in the grid they sit at opposite ends, 16 columns apart, so no kernel
  sees them together. At the row edges the kernel reaches past the grid, and padding
  fills those places with zeros.

`Folded` is two 2D kernels with a pool between them, flattened into a linear head. The 1D
arm to compare with is `hb.Kernels(3)` of 2.1.5: of the 1D stacks it has the number of
weights closest to the fold (the cell prints both, split into kernels and head), and it
is already trained.

In [ ]:
class Folded(nn.Module):
    # the beat folded to (B, 1, 11, 17), two 2D kernels with a pool, a linear head

    def __init__(self, width: int = 32):
        super().__init__()
        self.features = nn.Sequential(
            Conv2D(1, width, 3, padding=1),
            nn.GELU(),
            Pool2D(2),
            Conv2D(width, width, 3, padding=1),
            nn.GELU(),
        )
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 1, 11, 17)).flatten(1).shape[1]
        self.head = nn.Linear(flat, len(hb.CLASSES))

    def forward(self, x):
        grid = x[..., 0].reshape(-1, 1, 11, 17)  # (B, 187, 1) -> (B, 1, 11, 17)
        return self.head(self.features(grid).flatten(1))


draw(
    {
        "folded": [
            BEAT,
            Block("fold\nrow by row", "(B, 1, 11, 17)", "proj"),
            Block("kernel 3×3\n1 → 32, GELU", "(B, 32, 11, 17)", "hidden"),
            Block("max\npool 2×2", "(B, 32, 5, 8)", "noise"),
            Block("kernel 3×3\n32 → 32, GELU", "(B, 32, 5, 8)", "hidden"),
            Block("flatten", "(B, 1280)", "proj"),
            Block("Linear\n1280 → 3", "(B, 3)"),
        ]
    }
)
for name, model in (("folded", Folded()), ("depth 3 in 1D", hb.Kernels(3))):
    print(
        f"{name:14s} kernels {hb.n_params(model.features):6d}   head {hb.n_params(model.head):5d}   total {hb.n_params(model):6d}"
    )

grid = beat[:, 0].reshape(11, 17)
fig, axes = plt.subplots(1, 2, figsize=(12, 3), gridspec_kw={"width_ratios": [2, 1]})
axes[0].plot(t, beat[:, 0])
for row in range(1, 11):
    axes[0].axvline(row * 17 / 125, color="0.8", lw=0.8)
axes[0].set_title("(187, 1): the V beat, the rows of the fold marked")
axes[0].set_xlabel("seconds after the R-peak")
shown = axes[1].imshow(grid, cmap="viridis", aspect="auto")
axes[1].set_title("(1, 11, 17): the same 187 values")
axes[1].set_xlabel("step within the row")
axes[1].set_ylabel("row")
fig.colorbar(shown, ax=axes[1], label="scaled amplitude")
plt.show()

In the grid, a column holds steps 17 apart. Nothing in a beat repeats every 17 steps, so
the new neighbours are a bet without a reason behind it; it is tried here to see what the
shape alone does.

**Before you run the next cell**, write down what you expect from the fold against the 1D
stack of depth 3, and why.

In [ ]:
folded = hb.spread(Folded, train, valid, EPOCHS, REPEATS)
print(hb.line("depth 3 in 1D (2.1.5)", kernels["depth 3"], versus=floor))
print(hb.line("folded to (11, 17)", folded, versus=floor))
print(
    f"folded against depth 3: {hb.standard_errors(folded, kernels['depth 3']):+.1f} se"
)
print("recall per class, folded", {c: round(r, 2) for c, r in folded.recall().items()})
hb.curves(
    {"depth 3 in 1D": kernels["depth 3"], "folded": folded},
    "1D against folded; dashed: 1-nearest-neighbour",
    floor=floor.mean,
)

_Reading_. Both arms flatten by epoch 15 (the curves). The fold ends at 0.555 ± 0.017,
1.7 se above depth 3 in 1D: within about 2 se, so the difference is no result, whatever
its sign. Its recall of S is 2%, as in 1D. The fold is 3.4 se above 1-nearest-neighbour;
depth 3, at 2.1 se, is on the line.

This experiment does not show why the fold did not clearly help or hurt; it only shows
that it did not. What can be said is what the fold changes: which steps a kernel sees
together (steps 17 apart become neighbours) and the row wrap (steps 16 and 17 are cut
apart). As a hypothesis, not a finding: a fold would be expected to pay when the data
repeats with a period near the row width, so that the row above holds the same phase of
the previous period: a series of many beats folded by the length of one beat, a day of
ten-minute readings folded by 144, a year of months folded by 12. One cut-out beat has no
such period. You choose the shape; the data does not tell the kernel it was folded.

## What you now know

- **The floors and the encoder question (1.13, 2.9).** Without an encoder, the raw beat is
  its own vector. 1-nearest-neighbour on it reaches 0.526 and is the floor; a linear layer
  on the raw steps says N for nearly every beat, and lesson 1's head with a hidden layer
  ends below 1-nearest-neighbour at this training length. With 33,454 beats far from any
  photo or text, and a nearest raw beat that already scores well above 1/3, this lesson
  trains the whole model; with little data close to an encoder's pretraining data, you
  would reach for the encoder.
- **The metric.** The recall of a class is the share of it that was found; the macro recall
  is their mean, so S counts for a third. The plain loss does not see that.
- **A kernel (2.2).** A few weights, multiplied with the values under them and summed, the
  same at every position; you computed one by hand, with and without padding. A layer of
  `D'` kernels of size `k` on `D` channels has `D' × (D × k + 1)` weights, against 35,156
  for a dense layer from 187 steps to 187. On `(B, S, D)` and on `(B, C, H, W)` it is one
  operation; `Conv1D` takes the channels last.
- **Two kernels without an activation are one kernel (1.6)**, whose weights are the sums of
  products of theirs.
- **Fewer positions (2.3).** A stride learns how, a pool follows a fixed rule, a global
  pool removes the axis: `(8, 187, 16)` became `(8, 94, 16)`, `(8, 93, 16)` and `(8, 16)`.
- **The receptive field (2.4).** Start at 1; a kernel adds `(k − 1) × jump`, a pool of 2
  adds the jump and doubles it. With pools, 5 blocks see 156 steps; without, 21. With a
  flatten head every model reads the whole beat; the field is the span of one feature.
- **A kernel's bets (2.7).** Locality and weight sharing; the flatten head gives every
  position its own weights again. On these beats a wider kernel paid: one kernel of 35
  beat one of 5 at depth 1, and did at least as well as three or five layers, so depth was
  not needed for that gain. Whether the wider field or the extra kernel weights carried it,
  this notebook does not separate. S stayed at 3% or below in every arm and in
  1-nearest-neighbour; its rarity, its prematurity outside the window and a shift between
  patients all fit that.
- **A fold (2.8)** changes which steps a kernel sees together. On one beat it made no
  difference beyond about 2 se; when a fold should pay (data that repeats with the row
  width) is a hypothesis this notebook did not test.